In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/bannanakid/RCRL_on_RCMDP")
RUN_NUMBER = 1
EVALUATION_EPISODES = 10
EVALUATION_SEED = 2001
VIEWER_SEED = 1
VIEWER_STEPS = 1000
REALTIME_VIEWER = True
USE_STATE_NORM = False

os.environ["MUJOCO_GL"] = "glfw"
os.environ.pop("PYOPENGL_PLATFORM", None)
os.environ.pop("MUJOCO_PY_FORCE_CPU", None)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MODEL_DIRECTORY = PROJECT_ROOT / "models" / "HalfCheetahUnconstrained" / f"run{RUN_NUMBER}"
MODEL_CANDIDATES = [
    MODEL_DIRECTORY / "Best_Unconstrained_actor",
    MODEL_DIRECTORY / "Best_RCAC_actor",
]
MODEL_FILE = next((path for path in MODEL_CANDIDATES if path.exists()), None)

if MODEL_FILE is None:
    raise FileNotFoundError(
        "No actor file found. Expected one of:\n"
        + "\n".join(str(path) for path in MODEL_CANDIDATES)
    )

OUTPUT_DIRECTORY = (
    PROJECT_ROOT
    / "plot_data"
    / "HalfCheetahUnconstrained"
    / f"run{RUN_NUMBER}"
    / "evaluation"
)
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Model:", MODEL_FILE)
print("Output:", OUTPUT_DIRECTORY)

In [ ]:
import time
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from envs.half_cheetah_unconstrained import HalfCheetahUnconstrained

JOINT_NAMES = (
    "back_thigh",
    "back_shin",
    "back_foot",
    "front_thigh",
    "front_shin",
    "front_foot",
)

def unwrap_reset_observation(reset_result):
    result = reset_result
    while isinstance(result, tuple):
        result = result[0]
    return np.asarray(result, dtype=np.float64)

def unpack_step_result(step_result):
    if len(step_result) == 6:
        observation, reward, cost, truncated, terminated, info = step_result
        return observation, float(reward), float(cost), bool(truncated), bool(terminated), dict(info)
    if len(step_result) == 5:
        observation, reward, third, fourth, info = step_result
        if isinstance(third, (bool, np.bool_)) and isinstance(fourth, (bool, np.bool_)):
            return observation, float(reward), 0.0, bool(fourth), bool(third), dict(info)
        return observation, float(reward), float(third), bool(fourth), False, dict(info)
    if len(step_result) == 4:
        observation, reward, done, info = step_result
        return observation, float(reward), 0.0, False, bool(done), dict(info)
    raise ValueError(f"Unexpected environment step result length: {len(step_result)}")

class ActorGaussian(nn.Module):
    def __init__(self, state_dim, hidden_width, action_dim, max_action):
        super().__init__()
        self.max_action = float(max_action)
        self.fc1 = nn.Linear(state_dim, hidden_width)
        self.fc2 = nn.Linear(hidden_width, hidden_width)
        self.fc3 = nn.Linear(hidden_width, hidden_width)
        self.mean_layer = nn.Linear(hidden_width, action_dim)
        self.log_std = nn.Parameter(torch.zeros(1, action_dim))
        self.activate_func = nn.ReLU()

    def forward(self, state):
        state = self.activate_func(self.fc1(state))
        state = self.activate_func(self.fc2(state))
        state = self.activate_func(self.fc3(state))
        return self.max_action * torch.tanh(self.mean_layer(state))

def extract_state_dict(loaded):
    if isinstance(loaded, dict) and "state_dict" in loaded:
        return loaded["state_dict"]
    if isinstance(loaded, dict) and "actor_state_dict" in loaded:
        return loaded["actor_state_dict"]
    return loaded

In [ ]:
environment = HalfCheetahUnconstrained()
loaded_actor = torch.load(MODEL_FILE, map_location=torch.device("cpu"))
actor_state = extract_state_dict(loaded_actor)

state_dim = int(actor_state["fc1.weight"].shape[1])
hidden_width = int(actor_state["fc1.weight"].shape[0])
action_dim = int(actor_state["mean_layer.weight"].shape[0])
max_action = float(environment.action_space.high[0])

actor = ActorGaussian(state_dim, hidden_width, action_dim, max_action)
actor.load_state_dict(actor_state)
actor.eval()

state_norm = None
state_norm_file = MODEL_DIRECTORY / (
    "Best_Unconstrained_state_norm"
    if (MODEL_DIRECTORY / "Best_Unconstrained_state_norm").exists()
    else "Best_RCAC_state_norm"
)

if USE_STATE_NORM:
    if not state_norm_file.exists():
        raise FileNotFoundError(state_norm_file)
    with open(state_norm_file, "rb") as file:
        state_norm = pickle.load(file)

def prepare_state(observation):
    state = np.asarray(observation, dtype=np.float64)
    if state_norm is not None:
        state = state_norm(state, update=False)
    return state

def deterministic_action(observation):
    state = prepare_state(observation)
    state_tensor = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        action = actor(state_tensor).cpu().numpy().reshape(-1)
    return np.clip(action, environment.action_space.low, environment.action_space.high)

initial_observation = unwrap_reset_observation(environment.reset(seed=EVALUATION_SEED))
initial_action = deterministic_action(initial_observation)

print("State dimension:", state_dim)
print("Hidden width:", hidden_width)
print("Action dimension:", action_dim)
print("Initial observation shape:", initial_observation.shape)
print("Initial deterministic action:", initial_action)
print("Action finite:", bool(np.isfinite(initial_action).all()))

In [ ]:
step_rows = []
episode_rows = []

for episode_index in range(EVALUATION_EPISODES):
    reset_seed = EVALUATION_SEED + episode_index
    observation = unwrap_reset_observation(environment.reset(seed=reset_seed))
    starting_x = float(environment.sim.data.qpos[0])
    previous_x = starting_x
    episode_reward = 0.0
    episode_cost = 0.0
    traveled_distance = 0.0
    episode_pitch = []
    episode_joint_speeds = []
    episode_actions = []
    done = False
    step_index = 0

    while not done:
        action = deterministic_action(observation)
        next_observation, reward, cost, truncated, terminated, info = unpack_step_result(
            environment.step(action)
        )
        done = truncated or terminated
        current_x = float(info.get("xpos", environment.sim.data.qpos[0]))
        traveled_distance += abs(current_x - previous_x)
        previous_x = current_x

        torso_pitch = float(info.get("torso_pitch", environment.sim.data.qpos[2]))
        torso_pitch_abs = float(info.get("torso_pitch_abs", abs(torso_pitch)))
        joint_velocities = np.asarray(
            info.get("joint_velocities", environment.sim.data.qvel[3:9]),
            dtype=np.float64,
        )
        joint_speeds = np.asarray(
            info.get("joint_speeds", np.abs(joint_velocities)),
            dtype=np.float64,
        )

        row = {
            "episode": episode_index + 1,
            "seed": reset_seed,
            "step": step_index,
            "reward": reward,
            "cost": cost,
            "xpos": current_x,
            "torso_pitch": torso_pitch,
            "torso_pitch_abs": torso_pitch_abs,
            "maximum_joint_speed": float(np.max(joint_speeds)),
            "maximum_action_abs": float(np.max(np.abs(action))),
        }

        for joint_index, joint_name in enumerate(JOINT_NAMES):
            row[f"{joint_name}_velocity"] = float(joint_velocities[joint_index])
            row[f"{joint_name}_speed"] = float(joint_speeds[joint_index])
            row[f"action_{joint_name}"] = float(action[joint_index])

        step_rows.append(row)
        episode_reward += reward
        episode_cost += cost
        episode_pitch.append(torso_pitch_abs)
        episode_joint_speeds.append(joint_speeds)
        episode_actions.append(np.abs(action))
        observation = next_observation
        step_index += 1

    pitch_array = np.asarray(episode_pitch, dtype=np.float64)
    joint_speed_array = np.asarray(episode_joint_speeds, dtype=np.float64)
    action_array = np.asarray(episode_actions, dtype=np.float64)
    ending_x = float(environment.sim.data.qpos[0])

    episode_row = {
        "episode": episode_index + 1,
        "seed": reset_seed,
        "steps": step_index,
        "episode_reward": episode_reward,
        "episode_cost": episode_cost,
        "net_displacement": ending_x - starting_x,
        "absolute_displacement": abs(ending_x - starting_x),
        "traveled_distance": traveled_distance,
        "pitch_mean": float(np.mean(pitch_array)),
        "pitch_p95": float(np.percentile(pitch_array, 95)),
        "pitch_p99": float(np.percentile(pitch_array, 99)),
        "pitch_max": float(np.max(pitch_array)),
        "maximum_action_abs": float(np.max(action_array)),
    }

    for joint_index, joint_name in enumerate(JOINT_NAMES):
        episode_row[f"{joint_name}_speed_mean"] = float(np.mean(joint_speed_array[:, joint_index]))
        episode_row[f"{joint_name}_speed_p95"] = float(np.percentile(joint_speed_array[:, joint_index], 95))
        episode_row[f"{joint_name}_speed_p99"] = float(np.percentile(joint_speed_array[:, joint_index], 99))
        episode_row[f"{joint_name}_speed_max"] = float(np.max(joint_speed_array[:, joint_index]))

    episode_rows.append(episode_row)

step_data = pd.DataFrame(step_rows)
episode_data = pd.DataFrame(episode_rows)

display(episode_data.round(4))

print("Mean episode reward:", float(episode_data["episode_reward"].mean()))
print("Mean absolute displacement:", float(episode_data["absolute_displacement"].mean()))
print("Mean traveled distance:", float(episode_data["traveled_distance"].mean()))
print("Maximum recorded cost:", float(step_data["cost"].max()))
print("Finite values:", bool(np.isfinite(step_data.select_dtypes(include=[np.number]).to_numpy()).all()))

In [ ]:
summary_rows = [
    {
        "metric": "absolute_torso_pitch",
        "mean": float(step_data["torso_pitch_abs"].mean()),
        "p90": float(step_data["torso_pitch_abs"].quantile(0.90)),
        "p95": float(step_data["torso_pitch_abs"].quantile(0.95)),
        "p99": float(step_data["torso_pitch_abs"].quantile(0.99)),
        "maximum": float(step_data["torso_pitch_abs"].max()),
    }
]

for joint_name in JOINT_NAMES:
    values = step_data[f"{joint_name}_speed"]
    summary_rows.append(
        {
            "metric": joint_name,
            "mean": float(values.mean()),
            "p90": float(values.quantile(0.90)),
            "p95": float(values.quantile(0.95)),
            "p99": float(values.quantile(0.99)),
            "maximum": float(values.max()),
        }
    )

metric_summary = pd.DataFrame(summary_rows)
display(metric_summary.round(5))

step_csv = OUTPUT_DIRECTORY / "unconstrained_step_metrics.csv"
episode_csv = OUTPUT_DIRECTORY / "unconstrained_episode_metrics.csv"
summary_csv = OUTPUT_DIRECTORY / "unconstrained_metric_summary.csv"
raw_npz = OUTPUT_DIRECTORY / "unconstrained_evaluation_metrics.npz"

step_data.to_csv(step_csv, index=False)
episode_data.to_csv(episode_csv, index=False)
metric_summary.to_csv(summary_csv, index=False)

np.savez_compressed(
    raw_npz,
    episode=step_data["episode"].to_numpy(dtype=np.int64),
    seed=step_data["seed"].to_numpy(dtype=np.int64),
    step=step_data["step"].to_numpy(dtype=np.int64),
    reward=step_data["reward"].to_numpy(dtype=np.float64),
    cost=step_data["cost"].to_numpy(dtype=np.float64),
    xpos=step_data["xpos"].to_numpy(dtype=np.float64),
    torso_pitch=step_data["torso_pitch"].to_numpy(dtype=np.float64),
    torso_pitch_abs=step_data["torso_pitch_abs"].to_numpy(dtype=np.float64),
    joint_velocities=step_data[
        [f"{name}_velocity" for name in JOINT_NAMES]
    ].to_numpy(dtype=np.float64),
    joint_speeds=step_data[
        [f"{name}_speed" for name in JOINT_NAMES]
    ].to_numpy(dtype=np.float64),
    actions=step_data[
        [f"action_{name}" for name in JOINT_NAMES]
    ].to_numpy(dtype=np.float64),
    episode_rewards=episode_data["episode_reward"].to_numpy(dtype=np.float64),
    episode_costs=episode_data["episode_cost"].to_numpy(dtype=np.float64),
    net_displacements=episode_data["net_displacement"].to_numpy(dtype=np.float64),
    absolute_displacements=episode_data["absolute_displacement"].to_numpy(dtype=np.float64),
    traveled_distances=episode_data["traveled_distance"].to_numpy(dtype=np.float64),
    joint_names=np.asarray(JOINT_NAMES),
)

print(step_csv)
print(episode_csv)
print(summary_csv)
print(raw_npz)

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(step_data["torso_pitch_abs"], bins=80, density=True, alpha=0.65)
plt.xlabel("Absolute torso pitch")
plt.ylabel("Density")
plt.title("Unconstrained torso-pitch distribution")
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 6))
for joint_name in JOINT_NAMES:
    values = np.sort(step_data[f"{joint_name}_speed"].to_numpy())
    probabilities = np.arange(1, len(values) + 1) / len(values)
    plt.plot(values, probabilities, label=joint_name)
plt.xlabel("Absolute joint speed")
plt.ylabel("Empirical cumulative probability")
plt.title("Unconstrained joint-speed ECDF")
plt.legend()
plt.tight_layout()
plt.show()

selected_episode = int(
    episode_data.loc[episode_data["episode_reward"].idxmax(), "episode"]
)
selected_data = step_data[step_data["episode"] == selected_episode]

plt.figure(figsize=(11, 5))
plt.plot(selected_data["step"], selected_data["torso_pitch_abs"])
plt.xlabel("Step")
plt.ylabel("Absolute torso pitch")
plt.title(f"Torso pitch in highest-reward episode {selected_episode}")
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 6))
for joint_name in JOINT_NAMES:
    plt.plot(
        selected_data["step"],
        selected_data[f"{joint_name}_speed"],
        label=joint_name,
    )
plt.xlabel("Step")
plt.ylabel("Absolute joint speed")
plt.title(f"Joint speeds in highest-reward episode {selected_episode}")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
from mujoco_py import MjViewer

viewer_environment = HalfCheetahUnconstrained()
viewer_observation = unwrap_reset_observation(
    viewer_environment.reset(seed=VIEWER_SEED)
)
viewer = MjViewer(viewer_environment.sim)
viewer.cam.trackbodyid = 0
viewer.cam.distance = 5.0
viewer.cam.lookat[2] = 1.0
viewer.cam.elevation = -20.0

viewer_reward = 0.0
viewer_cost = 0.0
viewer_start_x = float(viewer_environment.sim.data.qpos[0])

for viewer_step in range(VIEWER_STEPS):
    viewer_action = deterministic_action(viewer_observation)
    (
        viewer_observation,
        viewer_step_reward,
        viewer_step_cost,
        viewer_truncated,
        viewer_terminated,
        viewer_info,
    ) = unpack_step_result(viewer_environment.step(viewer_action))
    viewer_reward += viewer_step_reward
    viewer_cost += viewer_step_cost
    viewer.render()
    if REALTIME_VIEWER:
        time.sleep(viewer_environment.dt)
    if viewer_truncated or viewer_terminated:
        break

viewer_end_x = float(viewer_environment.sim.data.qpos[0])

print("Viewer steps:", viewer_step + 1)
print("Viewer reward:", viewer_reward)
print("Viewer cost:", viewer_cost)
print("Viewer net displacement:", viewer_end_x - viewer_start_x)